In [5]:
import tensorflow as tf
from tensorflow.keras import layers, Model

In [6]:
# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────
NUM_FRAMES     = 150        # frames per video clip
VIDEO_H        = 224        # frame height
VIDEO_W        = 224        # frame width
VIDEO_C        = 3          # RGB channels
NUM_LANDMARKS  = 21         # MediaPipe hand landmarks
COORDS         = 2          # x, y per landmark
EMBED_DIM      = 128        # Transformer embedding dimension
NUM_HEADS      = 4          # Multi-head attention heads
FFN_DIM        = 256        # Feed-forward network inner dim
DROPOUT_RATE   = 0.3        # Global dropout rate
NUM_CLASSES    = 26         # Number of output classes (e.g. A–Z)

In [8]:
# ─────────────────────────────────────────────
# BLOCK 1 — 3D CNN Block
# Used 3× in the Video stream
# Input shape:  (batch, frames, H, W, C)
# Output shape: (batch, frames', H', W', filters)
# ─────────────────────────────────────────────
def cnn3d_block(x, filters: int, name_prefix: str):
    x = layers.Conv3D(
        filters, kernel_size=(3, 3, 3),
        padding="same", activation="relu",
        name=f"{name_prefix}_conv3d"
    )(x)
    x = layers.BatchNormalization(name=f"{name_prefix}_bn")(x)
    x = layers.Dropout(DROPOUT_RATE, name=f"{name_prefix}_dropout")(x)
    x = layers.MaxPooling3D(pool_size=(1, 2, 2), name=f"{name_prefix}_maxpool")(x)
    return x


# ─────────────────────────────────────────────
# BLOCK 2 — Transformer Encoder Block
# Input shape:  (batch, seq_len, embed_dim)
# Output shape: (batch, seq_len, embed_dim)
#
# Internal structure:
#   Multi-Head Attention
#   → Dropout → Residual Add → Layer Norm      (first sub-block)
#   → Dense FFN → Dropout → Residual Add       (second sub-block)
# ─────────────────────────────────────────────
def transformer_encoder_block(x, name_prefix: str):
    # ── Sub-block 1: Multi-Head Self-Attention ──
    attn_output = layers.MultiHeadAttention(
        num_heads=NUM_HEADS,
        key_dim=EMBED_DIM // NUM_HEADS,
        name=f"{name_prefix}_mha"
    )(x, x)                                                         # self-attention
    attn_output = layers.Dropout(DROPOUT_RATE,
        name=f"{name_prefix}_attn_dropout")(attn_output)
    x = layers.Add(name=f"{name_prefix}_attn_add")([x, attn_output])   # residual
    x = layers.LayerNormalization(name=f"{name_prefix}_attn_ln")(x)    # layer norm

    # ── Sub-block 2: Feed-Forward Network ──
    ffn_output = layers.Dense(FFN_DIM, activation="relu",
        name=f"{name_prefix}_ffn_dense1")(x)
    ffn_output = layers.Dense(EMBED_DIM,
        name=f"{name_prefix}_ffn_dense2")(ffn_output)
    ffn_output = layers.Dropout(DROPOUT_RATE,
        name=f"{name_prefix}_ffn_dropout")(ffn_output)
    x = layers.Add(name=f"{name_prefix}_ffn_add")([x, ffn_output])     # residual

    return x


# ─────────────────────────────────────────────
# STREAM 1 — Video Branch
# Input:  (batch, 150, 224, 224, 3)
# Output: (batch, seq_len, EMBED_DIM)
# ─────────────────────────────────────────────
def build_video_branch(video_input):
    x = video_input

    # 3D CNN Block ×3 — progressively extract spatiotemporal features
    x = cnn3d_block(x, filters=32,  name_prefix="cnn3d_1")  # → (150, 112, 112, 32)
    x = cnn3d_block(x, filters=64,  name_prefix="cnn3d_2")  # → (150,  56,  56, 64)
    x = cnn3d_block(x, filters=128, name_prefix="cnn3d_3")  # → (150,  28,  28, 128)

    # Reshape / Flatten spatial dims → treat frames as sequence
    # (batch, frames, H', W', C') → (batch, frames, H'*W'*C')
    _, t, h, w, c = x.shape
    x = layers.Reshape((t, h * w * c), name="video_reshape")(x)

    # Dense projection → embed to EMBED_DIM
    x = layers.Dense(EMBED_DIM, activation="relu", name="video_dense")(x)

    return x  # (batch, 150, EMBED_DIM)


# ─────────────────────────────────────────────
# STREAM 2 — Keypoint Branch
# Input:  (batch, 150, 42)   [21 landmarks × (x,y)]
# Output: (batch, seq_len, EMBED_DIM)
# ─────────────────────────────────────────────
def build_keypoint_branch(kp_input):
    # Reshape / Flatten — already flat (150, 42), kept for explicit clarity
    x = layers.Reshape((NUM_FRAMES, NUM_LANDMARKS * COORDS),
        name="kp_reshape")(kp_input)

    # Dense projection → embed to EMBED_DIM
    x = layers.Dense(EMBED_DIM, activation="relu", name="kp_dense")(x)

    # Dropout before Transformer
    x = layers.Dropout(DROPOUT_RATE, name="kp_dropout")(x)

    return x  # (batch, 150, EMBED_DIM)


# ─────────────────────────────────────────────
# FULL MODEL
# ─────────────────────────────────────────────
def build_model() -> Model:
    # ── Inputs ──
    video_input = tf.keras.Input(
        shape=(NUM_FRAMES, VIDEO_H, VIDEO_W, VIDEO_C),
        name="video_input"
    )
    kp_input = tf.keras.Input(
        shape=(NUM_FRAMES, NUM_LANDMARKS * COORDS),
        name="keypoint_input"
    )

    # ── Stream 1: Video ──
    video_features = build_video_branch(video_input)
    video_encoded  = transformer_encoder_block(video_features, name_prefix="video_transformer")

    # ── Stream 2: Keypoint ──
    kp_features = build_keypoint_branch(kp_input)
    kp_encoded  = transformer_encoder_block(kp_features, name_prefix="kp_transformer")

    # ── Concatenation ──
    # Both are (batch, 150, EMBED_DIM) → concat on last axis → (batch, 150, EMBED_DIM*2)
    merged = layers.Concatenate(axis=-1, name="concat")([video_encoded, kp_encoded])

    # Project back to EMBED_DIM after concat
    merged = layers.Dense(EMBED_DIM, activation="relu", name="merged_proj")(merged)

    # ── Final Transformer Encoder Block ──
    final_encoded = transformer_encoder_block(merged, name_prefix="final_transformer")

    # ── Global Average Pooling ──
    # (batch, 150, EMBED_DIM) → (batch, EMBED_DIM)
    pooled = layers.GlobalAveragePooling1D(name="gap")(final_encoded)

    # ── Classification Head ──
    output = layers.Dense(NUM_CLASSES, activation="softmax", name="output")(pooled)

    model = Model(
        inputs=[video_input, kp_input],
        outputs=output,
        name="VideoKeypoint_Transformer"
    )
    return model

In [9]:
# ─────────────────────────────────────────────
# MAIN — Build & Summarize
# ─────────────────────────────────────────────

model = build_model()
model.summary(expand_nested=True)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# ── Dummy input sanity check ──
import numpy as np
dummy_video = np.random.rand(2, NUM_FRAMES, VIDEO_H, VIDEO_W, VIDEO_C).astype("float32")
dummy_kp    = np.random.rand(2, NUM_FRAMES, NUM_LANDMARKS * COORDS).astype("float32")
dummy_label = np.array([0, 1])

output = model.predict([dummy_video, dummy_kp])
print(f"\n✅ Output shape: {output.shape}")   # Expected: (2, NUM_CLASSES)
print(f"✅ Sample prediction: {output[0]}")

Model: "VideoKeypoint_Transformer"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ video_input         │ (None, 150, 224,  │          0 │ -                 │
│ (InputLayer)        │ 224, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn3d_1_conv3d      │ (None, 150, 224,  │      2,624 │ video_input[0][0] │
│ (Conv3D)            │ 224, 32)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn3d_1_bn          │ (None, 150, 224,  │        128 │ cnn3d_1_conv3d[0… │
│ (BatchNormalizatio… │ 224, 32)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn3d_1_dropout     │ (None, 150, 224,  │          0 │ cnn3d_1_bn[0][0]  │
│ (Dropout)           │ 224, 32)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn3d_1_maxpool     │ (None, 150, 112,  │          0 │ cnn3d_1_dropout[… │
│ (MaxPooling3D)      │ 112, 32)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn3d_2_conv3d      │ (None, 150, 112,  │     55,360 │ cnn3d_1_maxpool[… │
│ (Conv3D)            │ 112, 64)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn3d_2_bn          │ (None, 150, 112,  │        256 │ cnn3d_2_conv3d[0… │
│ (BatchNormalizatio… │ 112, 64)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn3d_2_dropout     │ (None, 150, 112,  │          0 │ cnn3d_2_bn[0][0]  │
│ (Dropout)           │ 112, 64)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn3d_2_maxpool     │ (None, 150, 56,   │          0 │ cnn3d_2_dropout[… │
│ (MaxPooling3D)      │ 56, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn3d_3_conv3d      │ (None, 150, 56,   │    221,312 │ cnn3d_2_maxpool[… │
│ (Conv3D)            │ 56, 128)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn3d_3_bn          │ (None, 150, 56,   │        512 │ cnn3d_3_conv3d[0… │
│ (BatchNormalizatio… │ 56, 128)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn3d_3_dropout     │ (None, 150, 56,   │          0 │ cnn3d_3_bn[0][0]  │
│ (Dropout)           │ 56, 128)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ keypoint_input      │ (None, 150, 42)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn3d_3_maxpool     │ (None, 150, 28,   │          0 │ cnn3d_3_dropout[… │
│ (MaxPooling3D)      │ 28, 128)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ kp_reshape          │ (None, 150, 42)   │          0 │ keypoint_input[0… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ video_reshape       │ (None, 150,       │          0 │ cnn3d_3_maxpool[… │
│ (Reshape)           │ 100352)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ kp_dense (Dense)    │ (None, 150, 128)  │      5,504 │ kp_reshape[0][0]

 Total params: 13,563,802 (51.74 MB)

 Trainable params: 13,563,354 (51.74 MB)

 Non-trainable params: 448 (1.75 KB)

1/1 ━━━━━━━━━━━━━━━━━━━━ 17s 17s/step

✅ Output shape: (2, 26)
✅ Sample prediction: [0.09977271 0.00483537 0.04815033 0.04540247 0.00058193 0.05714077
 0.06167145 0.03534109 0.02943557 0.00698053 0.00214524 0.01514179
 0.00818433 0.02701309 0.01566291 0.04998625 0.0262677  0.00430723
 0.00791317 0.00491714 0.00657187 0.01896759 0.00179532 0.29590076
 0.00905688 0.11685651]
